### Lab 8.2 Part-of-Speech Tagging

In this lab you will experiment with creating a sequence-to-sequence model for [part-of-speech (POS)](https://en.wikipedia.org/wiki/Part-of-speech_tagging) tagging on the [Brown corpus](https://en.wikipedia.org/wiki/Brown_Corpus).

The Brown corpus consists of sentences tagged with parts of speech.  Here is an example:

*Sentence:*
The Fulton County grand jury said Friday an investigation of Atlanta's recent primary election produced ``no evidence'' that any irregularities took place .

*Tags:* DET NOUN NOUN ADJ NOUN VERB NOUN DET NOUN ADP NOUN ADJ NOUN NOUN VERB . DET NOUN . ADP DET NOUN VERB NOUN .





Download the dataset:

In [1]:
import os, urllib.request, zipfile

url = "https://www.dropbox.com/scl/fi/k5q12z1do2siqk1uri80f/brown_corpus.zip?rlkey=82z1akb1d0wacpvr7mje51khf&dl=1"
if not os.path.exists('brown_corpus'):
    urllib.request.urlretrieve(url, 'brown_corpus.zip')
    with zipfile.ZipFile('brown_corpus.zip') as z:
        z.extractall('.')


Load the data from pickle files:

In [2]:
import pickle

tokenized_text = pickle.load(open('brown_corpus/tokens.pkl','rb'))
labels = pickle.load(open('brown_corpus/labels.pkl','rb'))

`tokenized_text` is a list of lists of integers indicating the tokens in each sentence.

In [3]:
tokenized_text[0][:10]

[19304, 19707, 9944, 15509, 46802, 14723, 14582, 40619, 988, 27781]

`labels` is a corresponding list of lists of integers indicating the POS tags.

In [4]:
labels[0][:10]

[6, 3, 3, 7, 3, 9, 3, 6, 3, 4]

Calculate vocabulary size and number of labels from the dataset.

In [5]:
import numpy as np

vocab_size = int(np.max([np.max(t) for t in tokenized_text])+1)

num_labels = int(np.max([np.max(l) for l in labels])+1)
print(f'Vocabulary size: {vocab_size}\tNumber of labels: {num_labels}')

Vocabulary size: 49815	Number of labels: 13


Make a 90/10 train/test split of the dataset.

In [6]:
from sklearn.model_selection import train_test_split
tokenized_text_train, tokenized_text_test, labels_train, labels_test = train_test_split(tokenized_text,labels,test_size=0.1,random_state=42)

Here is code for a custom `Dataset` subclass that produces subsequences of the dataset.

In [7]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]
    label_ids = self.labels[idx]

    # crop or pad as necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        # choose random substring
        ind = np.random.randint(len(token_ids)-self.max_seq_len)
        token_ids = token_ids[ind:ind+self.max_seq_len]
        label_ids = label_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
        label_ids = [0]*(self.max_seq_len-len(label_ids)) + label_ids
    
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(label_ids).long()


Create datasets and data loaders:

In [8]:
train_ds = TokenDataset(tokenized_text_train,labels_train,max_seq_len=10)
test_ds = TokenDataset(tokenized_text_test,labels_test)

train_dl = DataLoader(train_ds,shuffle=True,batch_size=32)
test_dl = DataLoader(test_ds,shuffle=False,batch_size=1)

### Exercises

1. Grab a batch of data from `train_dl`.  Inspect the data (the shapes and values) and explain what you see.

In [9]:
x_batch, y_batch = next(iter(train_dl))

In [10]:
print('x_batch shape:', x_batch.shape, x_batch.dtype)
print('y_batch shape:', y_batch.shape, y_batch.dtype)
print('First sequence tokens:', x_batch[0])
print('First sequence labels:', y_batch[0])

x_batch shape: torch.Size([32, 10]) torch.int64
y_batch shape: torch.Size([32, 10]) torch.int64
First sequence tokens: tensor([15993, 26655, 13638, 14944, 47812, 41590, 40706, 21595, 35985, 35996])
First sequence labels: tensor([ 9, 12,  4,  6,  3,  3,  3,  3, 10,  3])


I see a lot of words / tkens within the data. However the shape of the data is quite small compared to the actual size of the inputs. Moreover our labels are few compared to what we have. 

2. Train a RNN model (from lab 8.1) for the POS tagging task.  Report accuracy on the test set.

Notes:
* Before the RNN, you need to use an ``nn.Embedding`` model to map tokens to vectors.
* `torch.nn.CrossEntropyLoss` and `torchmetrics.classification.Accuracy` cannot handle sequence inputs.  You will need to combine the batch and sequence dimensions using `torch.flatten` before computing the loss or computing accuracy.

In [11]:
from torchmetrics.classification import Accuracy

class RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
      super().__init__()
      self.hidden_size = hidden_size
      self.U = nn.Linear(input_size, hidden_size)
      self.W = nn.Linear(hidden_size, hidden_size)
      self.V = nn.Linear(hidden_size, output_size)

    def forward(self,x):
      B, N, C = x.shape
      h = torch.zeros(B, self.hidden_size)
      outputs = []
      for t in range(N):
          h = torch.relu(self.U(x[:, t]) + self.W(h))
          outputs.append(self.V(h))
      return torch.stack(outputs, dim=1)

class POSTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_labels):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = RNN(embed_dim, hidden_size, num_labels)

    def forward(self, x):
        return self.rnn(self.embedding(x))

torch.manual_seed(42)
model = POSTagger(vocab_size, embed_dim=64, hidden_size=128, num_labels=num_labels)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        logits = model(xb)
        loss = loss_fn(torch.flatten(logits, 0, 1), torch.flatten(yb))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}: loss {total_loss/len(train_dl):.4f}')

# Test accuracy
acc = Accuracy(task='multiclass', num_classes=num_labels, ignore_index=0)
model.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        acc.update(torch.flatten(model(xb), 0, 1), torch.flatten(yb))
print(f'Test accuracy: {acc.compute().item():.4f}')


Epoch 1: loss 0.5723
Epoch 2: loss 0.2746
Epoch 3: loss 0.2007
Epoch 4: loss 0.1649
Epoch 5: loss 0.1396
Test accuracy: 0.9432
